# Join Harmonized DOC Data with TopoCat Catchments

This notebook loads the harmonized DOC dataset, finds the nearest catchment geometry from TopoCat, and assigns catchment geometry to each DOC sample. The results are written to a file for downstream analysis.

## 1. Import Required Libraries

In [1]:
import geopandas as gpd
import pandas as pd

from land_cover.load import load_harmonized_doc, load_topocat_catchments, doc_jn_catchment_pth

%load_ext autoreload
%autoreload 2

In [25]:
# Temp: for missing catchments, assuming largest missed lake is 0.02 km2 or 200x100 m
buffer_len = 300  # in meters

## 2. Load and Execute the Main Function

In [37]:
# Load the harmonized DOC dataset
print("Loading harmonized DOC data...")
gdf_doc = load_harmonized_doc()
print(f"Loaded {len(gdf_doc)} DOC records")
gdf_doc.info(show_counts=True, max_cols=5000)

Loading harmonized DOC data...
Loaded 3808 DOC records
<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 3808 entries, 0 to 4294
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype   
---  ------      --------------  -----   
 0   lat         3808 non-null   float64 
 1   lon         3808 non-null   float64 
 2   sample_id   3808 non-null   object  
 3   sample_idx  3808 non-null   object  
 4   area_km2    1513 non-null   object  
 5   doc         3568 non-null   float64 
 6   dic         1607 non-null   float64 
 7   source      3808 non-null   object  
 8   geometry    3808 non-null   geometry
dtypes: float64(4), geometry(1), object(4)
memory usage: 297.5+ KB


## 3. Load Catchment Data

In [3]:
# Load TopoCat catchment data
print("Loading TopoCat catchments...")
df_cat = load_topocat_catchments()
print(f"Loaded {len(df_cat)} catchments")
print(f"CRS: {df_cat.crs}")
print(f"Columns: {list(df_cat.columns)[:10]}...")  # Show first 10 columns

Loading TopoCat catchments...


/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


Loaded 3390139 catchments
CRS: EPSG:4326
Columns: ['Outlet_id_tpcat', 'lake_id_tpcat', 'D_out_id_tpcat', 'D_lake_id_tpcat', 'Cat_area_tpcat', 'Cat_type_tpcat', 'Basin_id_tpcat', 'Shape_Length_tpcat', 'Shape_Area_tpcat', 'geometry']...


## 4. Find Nearest Catchment Geometry

In [38]:
# Ensure both datasets are in the same CRS for spatial operations
if gdf_doc.crs != df_cat.crs:
    print(f"Reprojecting DOC data from {gdf_doc.crs} to {df_cat.crs}")
    gdf_doc_reprojected = gdf_doc.to_crs(df_cat.crs)
else:
    gdf_doc_reprojected = gdf_doc.copy()

# Find nearest catchment for each DOC sample using spatial join
print("\nFinding nearest catchment for each DOC sample...")
joined = gpd.sjoin_nearest(gdf_doc_reprojected, df_cat, how='left', max_distance=0.01)

# Just in case multiple right matches: de-duplicate
joined = joined.groupby("sample_idx", as_index=False).first()
joined.crs = df_cat.crs

# save right geoms
nearest_indices = joined.index_right.dropna()
nearest_geoms = df_cat.loc[nearest_indices, "geometry"].values

print(f"Found nearest catchments for {len(nearest_geoms)} / {len(gdf_doc_reprojected)} DOC samples")


Finding nearest catchment for each DOC sample...


/Users/ekyzivat/mambaforge/envs/landcover/lib/python3.11/site-packages/geopandas/array.py:403: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


Found nearest catchments for 3190 / 3808 DOC samples


In [39]:
joined

,sample_idx,lat,lon,sample_id,area_km2,doc,dic,source,geometry,index_right,Outlet_id_tpcat,lake_id_tpcat,D_out_id_tpcat,D_lake_id_tpcat,Cat_area_tpcat,Cat_type_tpcat,Basin_id_tpcat,Shape_Length_tpcat,Shape_Area_tpcat
0,D0,79.95000,-84.33000,1.0,0.2,20.30,51.8,Dranga17,POINT (-84.33 79.95),NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
1,D1,79.95000,-84.33000,2.0,0.1,21.00,52.9,Dranga17,POINT (-84.33 79.95),NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
2,D10,82.10000,-68.62000,11.0,192.8,4.10,10.8,Dranga17,"MULTIPOLYGON (((-68.64707 82.10908, -68.64706 ...",NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
3,D100,75.09000,-93.60000,101.0,None,1.10,18.8,Dranga17,"MULTIPOLYGON (((-93.69808 75.0933, -93.69704 7...",1913360.0,860007554.0,8.620019e+09,0.0,0.000000e+00,2.164422,Flow Through,860001230.0,0.208333,0.000675
4,D1000,76.45000,-119.41000,1001.0,None,3.00,4.7,Dranga17,POINT (-119.41 76.45),NaN,NaN,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3803,S995,66.38201,-164.04045,NSP16-NKM-W1,None,5.72,NaN,Stolpmann21,POINT (-164.04045 66.38201),172253.0,810259028.0,8.130059e+09,810259757.0,8.130064e+09,20.306432,Headwater,810004032.0,0.385000,0.004069
3804,S996,66.34602,-164.11681,NSP16-SKM-W1,None,1.68,NaN,Stolpmann21,POINT (-164.11681 66.34602),173492.0,810260900.0,8.130072e+09,0.0,0.000000e+00,20.558609,Headwater,810004038.0,0.381667,0.004114
3805,S997,66.39273,-164.27458,NSP16-WFM-W1,None,10.17,NaN,Stolpmann21,POINT (-164.27458 66.39273),172650.0,810259620.0,8.130063e+09,810261369.0,8.130075e+09,5.713622,Headwater,810004027.0,0.205000,0.001146
3806,S998,66.54679,-164.44714,NSP16-PRP-W1,None,23.32,NaN,Stolpmann21,POINT (-164.44714 66.54679),172011.0,810258638.0,8.130057e+09,810260966.0,8.130072e+09,1.600076,Headwater,810004036.0,0.096667,0.000323


## 5. Assign Geometry to DataFrame

In [40]:
# Create a new GeoDataFrame with catchment geometry and nearest catchment index
gdf_doc_jn_cat = joined.copy()
# gdf_doc_jn_cat["nearest_outlet_idx"] = "Outlet_id"
gdf_doc_jn_cat.loc[gdf_doc_jn_cat.index_right.notna(), "geometry"] = nearest_geoms
# gdf_doc_jn_cat = gpd.GeoDataFrame(
#     gdf_doc_jn_cat.drop(columns="index_right"), geometry="geometry", crs=df_cat.crs
# )
# For records without a nearest catchment match, create a buffer circle around the point
mask = gdf_doc_jn_cat.index_right.isna()
if mask.any():
    gdf_doc_jn_cat.loc[mask, "geometry"] = (
        gdf_doc_jn_cat.loc[mask].to_crs("ESRI:102001")
        .geometry.buffer(buffer_len)
        .to_crs("EPSG:4326")
    )
gdf_doc_jn_cat.drop(columns="index_right", inplace=True)
print(f"Created joined dataset with {len(gdf_doc_jn_cat)} records")
gdf_doc_jn_cat.info()

Created joined dataset with 3808 records
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 3808 entries, 0 to 3807
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   sample_idx          3808 non-null   object  
 1   lat                 3808 non-null   float64 
 2   lon                 3808 non-null   float64 
 3   sample_id           3808 non-null   object  
 4   area_km2            1513 non-null   object  
 5   doc                 3568 non-null   float64 
 6   dic                 1607 non-null   float64 
 7   source              3808 non-null   object  
 8   geometry            3808 non-null   geometry
 9   Outlet_id_tpcat     3190 non-null   float64 
 10  lake_id_tpcat       3190 non-null   float64 
 11  D_out_id_tpcat      3190 non-null   float64 
 12  D_lake_id_tpcat     3190 non-null   float64 
 13  Cat_area_tpcat      3190 non-null   float64 
 14  Cat_type_tpcat      3190 non-null   obj

## 6. Write Output to File

In [41]:
# Write to GeoPackage (supports polygons and all attributes)
print(f"Writing output to {doc_jn_catchment_pth}")
doc_jn_catchment_pth.parent.mkdir(parents=True, exist_ok=True)
gdf_doc_jn_cat.to_file(doc_jn_catchment_pth, driver="GPKG")

print(f"✓ Successfully wrote {len(gdf_doc_jn_cat)} records to {doc_jn_catchment_pth.name}")
print(f"\nOutput file info:")
print(f"  Path: {doc_jn_catchment_pth}")
print(f"  Size: {doc_jn_catchment_pth.stat().st_size / 1e6:.2f} MB")
print(f"  Records: {len(gdf_doc_jn_cat)}")
print(f"  CRS: {gdf_doc_jn_cat.crs}")

Writing output to /Volumes/metis/ABOVE3/Digitizing/catchments/doc_jn_catchments.gpkg
✓ Successfully wrote 3808 records to doc_jn_catchments.gpkg

Output file info:
  Path: /Volumes/metis/ABOVE3/Digitizing/catchments/doc_jn_catchments.gpkg
  Size: 56.35 MB
  Records: 3808
  CRS: EPSG:4326
✓ Successfully wrote 3808 records to doc_jn_catchments.gpkg

Output file info:
  Path: /Volumes/metis/ABOVE3/Digitizing/catchments/doc_jn_catchments.gpkg
  Size: 56.35 MB
  Records: 3808
  CRS: EPSG:4326
